In [1]:
import sys
sys.path.append('..')

import os
os.environ["KERAS_BACKEND"] = "torch"

import time
import keras
import numpy as np
# Necesario para TextVectorization y tf.data.
import tensorflow as tf
from models.training import compile_model, get_callbacks
from config.settings import Settings
from features.embeddings import load_gensim_embeddings
from datasets.dataset import create_dataset
from features.vectorizer import build_vectorizer
from datasets.loader import load_splits, save_json
from models.siamese_lstm import SiameseLSTM
from datasets.paths import ProjectPaths
from gensim.models import KeyedVectors


In [2]:
print(keras.config.backend())


torch


In [5]:
settings = Settings()
print(settings)


batch_size=64 mlp_dropout=0.4 lstm_dropout=0.3 pooling='mean' similarity='cosine' hidden_dim=64 bidirectional=False mlp_layers=[32] concat_features=['diff'] epochs=20 augmented_data=False siamese_name='lstm_mean_cosine_noaug_uncased'


In [6]:
paths = ProjectPaths(siamese_name=settings.siamese_name)


In [7]:
if settings.augmented_data:
	max_len = 27
	train_dir = paths.augmented_dir
else:
	max_len = 26
	train_dir = paths.processed_dir

print(max_len)

26


In [8]:
splits = {
	"train": train_dir,
	"dev": paths.processed_dir,
    "test": paths.processed_dir
}

datasets = load_splits(splits)

train_df = datasets["train"]
dev_df = datasets["dev"]
test_df = datasets["test"]


In [9]:
print("Train length:", len(train_df))
print("Dev length:", len(dev_df))


Train length: 5741
Dev length: 1497


In [10]:
all_sentences = list(train_df["sentence1"]) + list(train_df["sentence2"])

vectorizer = build_vectorizer(all_sentences, max_len)

vocab = vectorizer.get_vocabulary()
word2idx = {word: idx for idx, word in enumerate(vocab)}
print(f"Vocabulary size: {len(vocab)}")

vectorizer_model = keras.Sequential([vectorizer])
vectorizer_model.save(paths.vectorizer_path)


Vocabulary size: 12756


c:\Users\malos\Documents\GitHub\JustShare\server\.venv\Lib\site-packages\keras\src\saving\saving_api.py:107: UserWarning: You are saving a model that has not yet been built. It might not contain any weights yet. Consider building the model first by calling it on some data.
  return saving_lib.save_model(model, filepath)


In [11]:
wv = KeyedVectors.load_word2vec_format(paths.word2vec_path, binary=True)


In [12]:
dim = wv.vector_size
print(dim)


400


In [13]:
embedding_matrix = load_gensim_embeddings(wv, word2idx, dim)

print(embedding_matrix.shape)

np.save(paths.embedding_path, embedding_matrix)


Found 12096/12756 words
(12756, 400)


In [14]:
train_dataset = create_dataset(train_df, vectorizer, settings.batch_size, shuffle=True)
dev_dataset = create_dataset(dev_df, vectorizer, settings.batch_size)


In [15]:
for (sent1, sent2), y in train_dataset.take(1):
	print("sent1:", sent1.shape)
	print("sent2:", sent2.shape)
	print("y:", y.shape)


sent1: (64, 26)
sent2: (64, 26)
y: (64,)


In [16]:
vocab = vectorizer.get_vocabulary()
print(vocab[:10])


['', '[UNK]', np.str_('de'), np.str_('la'), np.str_('el'), np.str_('en'), np.str_('un'), np.str_('a'), np.str_('n'), np.str_('una')]


In [17]:
model = SiameseLSTM(
	vocab_size=len(vocab),
	embedding_dim=dim,
	hidden_dim=settings.hidden_dim,
	mlp_dropout=settings.mlp_dropout,
	lstm_dropout=settings.lstm_dropout,
	embedding_matrix=embedding_matrix,
	pooling=settings.pooling,
	similarity=settings.similarity,
	mlp_layers=settings.mlp_layers,
	bidirectional=settings.bidirectional,
	concat_features=settings.concat_features,
    name=settings.siamese_name
)


In [18]:
if model.mlp:
	model.mlp.summary()


In [19]:
head_model = model.get_head_model()
head_model.summary()


Model: "siamese_head"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_1         │ (None, None)      │          0 │ input_layer[0][0] │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 400) │  5,102,400 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cast (Cast)         │ (None, None)      │          0 │ not_equal_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, None, 64)  │    119,040 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expand_dims         │ (None, None, 1)   │          0 │ cast[0][0]        │
│ (ExpandDims)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, None, 64)  │          0 │ lstm[0][0],       │
│                     │                   │            │ expand_dims[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sum_1 (Sum)         │ (None, 1)         │          0 │ expand_dims[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sum (Sum)           │ (None, 64)        │          0 │ multiply[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 1)         │          0 │ sum_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ true_divide         │ (None, 64)        │          0 │ sum[0][0],        │
│ (TrueDivide)        │                   │            │ add[0][0]         │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 5,221,440 (19.92 MB)

 Trainable params: 119,040 (465.00 KB)

 Non-trainable params: 5,102,400 (19.46 MB)

In [20]:
dummy_sent1 = tf.zeros((1, max_len), dtype=tf.int32)
dummy_sent2 = tf.zeros((1, max_len), dtype=tf.int32)

model((dummy_sent1, dummy_sent2))

model.summary()


Model: "lstm_mean_cosine_noaug_uncased"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, None, 400)      │     5,102,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, None, 64)       │       119,040 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,221,440 (19.92 MB)

 Trainable params: 119,040 (465.00 KB)

 Non-trainable params: 5,102,400 (19.46 MB)

In [21]:
model = compile_model(model)

callbacks = get_callbacks(paths.siamese_path)


In [22]:
weights = model.embedding.get_weights()[0]

for i in range(10):
    print(i, vocab[i], weights[i][:5])
    

0  [0. 0. 0. 0. 0.]
1 [UNK] [-0.25303817  0.09669964  0.02313057  0.1977714   0.21378528]
2 de [ 0.21769126 -2.2936897  -1.3649052  -2.588661    2.8798313 ]
3 la [-1.8710854  -0.46249285  0.8997163  -0.17077468  2.4137783 ]
4 el [ 0.40440384 -2.3295271  -5.4817753   0.2512127   0.8362291 ]
5 en [ 1.6279118e+00  1.9744599e-04 -4.8397598e+00 -1.7532663e-01
  3.0406034e+00]
6 un [ 1.8357593   0.28429332 -3.9700549   0.31613564  1.1295024 ]
7 a [-1.2744071   0.12961593 -1.9868547   0.04239245  3.7093022 ]
8 n [ 0.84372216  1.2837858   1.2043453  -0.27897033 -1.7948582 ]
9 una [-0.02631674  1.976948    0.06162561 -1.074304    1.8378303 ]


In [23]:
start_time = time.perf_counter()

history = model.fit(
	train_dataset,
	validation_data=dev_dataset,
	epochs=settings.epochs,
	callbacks=callbacks
)

train_time = time.perf_counter() - start_time

np.save(paths.history_path, history.history)


Epoch 1/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 52s 576ms/step - loss: 0.0730 - mae: 0.2229 - rmse: 0.2702 - val_loss: 0.1264 - val_mae: 0.2897 - val_rmse: 0.3555 - learning_rate: 0.0010
Epoch 2/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 49s 547ms/step - loss: 0.0620 - mae: 0.2040 - rmse: 0.2490 - val_loss: 0.1052 - val_mae: 0.2624 - val_rmse: 0.3243 - learning_rate: 0.0010
Epoch 3/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 52s 577ms/step - loss: 0.0554 - mae: 0.1921 - rmse: 0.2353 - val_loss: 0.0903 - val_mae: 0.2412 - val_rmse: 0.3004 - learning_rate: 0.0010
Epoch 4/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 50s 557ms/step - loss: 0.0497 - mae: 0.1818 - rmse: 0.2230 - val_loss: 0.0868 - val_mae: 0.2352 - val_rmse: 0.2947 - learning_rate: 0.0010
Epoch 5/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 51s 562ms/step - loss: 0.0475 - mae: 0.1771 - rmse: 0.2179 - val_loss: 0.0883 - val_mae: 0.2369 - val_rmse: 0.2971 - learning_rate: 0.0010
Epoch 6/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 50s 553ms/step - loss: 0.0424 - mae: 0.1665 - rmse: 0.2058 - val_loss: 0.0788

In [24]:
run_config = {
    "sequence_length": max_len,
    "data_augmentation": settings.augmented_data,
    "train_time_s": train_time
}

save_json(run_config, paths.config_path)
